In [24]:
# For later: RAD - https://github.com/ubc-systopia/dsn-2022-rad-artifact/tree/main

import pandas as pd
import glob
import os

data_paths = ["../data/rad/known_procedures/benign","../data/rad/known_procedures/anomaly", "../data/rad/unknown_procedures"]

files = []
for data_path in data_paths:
    files += sorted(glob.glob(os.path.join(data_path, "*.csv")))


# files = sorted(glob.glob(os.path.join(data_paths[0], "*.csv")))

# Load and concatenate
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


In [25]:
df.head()

,Timestamp,Module,Method_Name,Arguments,Responses,Exceptions,id,Execution Time (Sec),Arrival_Time,Departure_Time
0,2021:10:12:13:22:23.845243,C9,_init_,ftdi: None,NaN,NaN,NaN,NaN,NaN,NaN
1,2021:10:12:13:22:24.499940,C9,PING,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021:10:12:13:22:24.799885,C9,BIAS,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021:10:12:13:22:25.072611,C9,SPED,"velocity: 20000, acceleration: 20000",NaN,NaN,NaN,NaN,NaN,NaN
4,2021:10:12:13:22:25.453128,C9,BIAS,bias: 0,NaN,NaN,NaN,NaN,NaN,NaN


These data files contain lots of data that is not related to the robot, including the status of other machines and unrelated events. We will ignore the extra stuff.

In [26]:
df.drop(columns=['Module', 'Responses', "Exceptions", 'Arrival_Time', 'Departure_Time', 'Execution Time (Sec)', 'id'], inplace=True)
df.head()

,Timestamp,Method_Name,Arguments
0,2021:10:12:13:22:23.845243,_init_,ftdi: None
1,2021:10:12:13:22:24.499940,PING,NaN
2,2021:10:12:13:22:24.799885,BIAS,NaN
3,2021:10:12:13:22:25.072611,SPED,"velocity: 20000, acceleration: 20000"
4,2021:10:12:13:22:25.453128,BIAS,bias: 0


In [27]:
df = df[df["Method_Name"].str.contains("ARM", case=False, na=False)]
df = df[~df["Arguments"].str.contains("velocity", case=False, na=False)]
df.drop(columns=['Method_Name'], inplace=True)
df.reset_index(drop=True, inplace=True)

df

,Timestamp,Arguments
0,2021:10:12:14:35:31.590194,"X: 159525, Y: 182500, Z: 187000, gripper: 1089..."
1,2021:10:12:14:35:32.141240,"X: 160151, Y: 182500, Z: 187000, gripper: 1089..."
2,2021:10:12:14:35:32.717996,"X: 160776, Y: 182500, Z: 187000, gripper: 1089..."
3,2021:10:12:14:35:33.188219,"X: 161401, Y: 182500, Z: 187000, gripper: 1089..."
4,2021:10:12:14:35:33.536488,"X: 162027, Y: 182500, Z: 187000, gripper: 1089..."
...,...,...
11098,2021:10:22:14:57:07.749848,"X: 252600, Y: -123400, Z: 290070, gripper: 785..."
11099,2021:10:22:14:57:07.749848,"X: 252600, Y: -123400, Z: 290070, gripper: 785..."
11100,2021:10:22:14:57:07.749848,"X: 252600, Y: -123400, Z: 290070, gripper: 785..."
11101,2021:10:22:14:57:07.749848,"X: 252600, Y: -123400, Z: 290070, gripper: 785..."


The only useful robot data is postion data.

In [28]:
# Encoding timestamps

# Convert to datetime
df['Timestamp'] = df['Timestamp'].astype(str).str.strip('"').str.strip("'").str.strip()

# Clean and normalize timestamp format
df['Timestamp'] = (
    df['Timestamp']
    .astype(str)
    .str.strip('"')
    .str.strip("'")
    .str.strip()
    .str.replace(":", "-", 2)     # Replace first two colons (Y:M:D → Y-M-D)
    .str.replace(":", "T", 1)     # Replace next colon (between day and hour) with T
)

df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='ISO8601', utc=True)

# Extract components, only use hour or shorter since all done in a single day
df['hour'] = df['Timestamp'].dt.hour
df['minute'] = df['Timestamp'].dt.minute
df['second'] = df['Timestamp'].dt.second
df['microsecond'] = df['Timestamp'].dt.microsecond

df.drop(columns=['Timestamp'], inplace=True)

df['time'] = (
    df['hour'] * 3600 + df['minute'] * 60 + df['second'] + df['microsecond'] / 1_000_000
)
df = df.sort_values('time')
df['time'] =  df['time'] - df['time'].iloc[0]

df.drop(columns=['hour', 'minute', 'second', 'microsecond'], inplace=True)

df.reset_index(drop=True, inplace=True)

df

,Arguments,time
0,"X: -186053, Y: -13900, Z: 250000, gripper: 150...",0.000000
1,"X: -181287, Y: -4773, Z: 250000, gripper: 1506...",0.846160
2,"X: -174021, Y: 2766, Z: 250000, gripper: 15066...",1.290711
3,"X: -165193, Y: 8004, Z: 250000, gripper: 15066...",1.717405
4,"X: -156365, Y: 13004, Z: 250000, gripper: 1506...",2.163429
...,...,...
11098,"X: -154900, Y: 212000, Z: 250000, gripper: 150...",16532.287049
11099,"X: -144900, Y: 212714, Z: 250000, gripper: 150...",16532.646601
11100,"X: -134900, Y: 213429, Z: 250000, gripper: 150...",16533.064527
11101,"X: -124900, Y: 214619, Z: 250000, gripper: 150...",16533.480524


In [29]:
# Extract X, Y, Z values using regex
df[['x', 'y', 'z']] = df['Arguments'].str.extract(
    r'X:\s*(-?\d+),\s*Y:\s*(-?\d+),\s*Z:\s*(-?\d+)'
).astype(int)

df.drop(columns=['Arguments'], inplace=True)

df[['x', 'y', 'z']] = df[['x', 'y', 'z']].astype(float) / 1000000  # Convert micrometers to meters

In [30]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def time_series_plots(df, error=None):
    # Define axis list and colors
    feature_type_lst = ["x", "y", "z"]
    colors = px.colors.qualitative.Dark24[:3]

    # Create 3 subplots for X, Y, Z
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        subplot_titles=("X Position", "Y Position", "Z Position"),
        vertical_spacing=0.08
    )

    # Loop over X, Y, Z
    for idx, feature_type in enumerate(feature_type_lst, start=1):
        if feature_type in df.columns:
            fig.add_trace(
                go.Scatter(
                    x=df['time'],
                    y=df[feature_type],
                    name=feature_type.upper(),
                    mode='lines',
                    line=dict(color=colors[idx - 1], width=2)
                ),
                row=idx, col=1
            )

    # Update axes and layout
    fig.update_xaxes(title_text="Time (s)", row=3, col=1, rangeslider_visible=True)
    for i, feature_type in enumerate(feature_type_lst, start=1):
        fig.update_yaxes(title_text=f"{feature_type} (pos)", row=i, col=1)

    fig.update_layout(
        height=900,
        title_text="Position Time Series (X, Y, Z)",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    return fig

# Example usage:
fig = time_series_plots(df)
fig.show()


In [31]:
import numpy as np
import pandas as pd

import numpy as np
import pandas as pd

def report_time_gaps(df, time_col='time', gap_threshold=2.0):
    """
    Prints the size of time gaps exceeding a threshold and reports the longest contiguous segment.

    Parameters:
        df : pd.DataFrame
            Must contain a numeric or datetime time column.
        time_col : str
            Name of the time column.
        gap_threshold : float
            Threshold (in seconds) for gap detection.
    """
    df = df.copy()

    # Convert to numeric seconds if datetime
    if not pd.api.types.is_numeric_dtype(df[time_col]):
        df[time_col] = pd.to_datetime(df[time_col])
        df[time_col] = (df[time_col] - df[time_col].iloc[0]).dt.total_seconds()

    # Sort by time
    df = df.sort_values(time_col).reset_index(drop=True)

    # Compute time differences
    time_vals = df[time_col].to_numpy()
    gaps = np.diff(time_vals)

    print(f"\nChecking for gaps > {gap_threshold} seconds...\n")

    gap_indices = np.where(gaps > gap_threshold)[0]
    gap_sizes = gaps[gaps > gap_threshold]

    if len(gap_sizes) == 0:
        print("No gaps exceeding threshold found.")
        print(f"Longest contiguous segment: {time_vals[0]:.3f}s to {time_vals[-1]:.3f}s "
              f"({time_vals[-1] - time_vals[0]:.3f} seconds long)")
        return

    # Print all gaps
    for gap in gap_sizes:
        print(f"Gap size: {gap:.3f} seconds")

    print(f"\nTotal gaps found: {len(gap_sizes)}")

    # Find segment boundaries between gaps
    segment_starts = np.concatenate(([0], gap_indices + 1))
    segment_ends = np.concatenate((gap_indices + 1, [len(time_vals)]))

    # Compute segment lengths
    segment_lengths = [time_vals[end - 1] - time_vals[start] for start, end in zip(segment_starts, segment_ends)]

    # Identify the longest segment
    longest_idx = np.argmax(segment_lengths)
    start_idx, end_idx = segment_starts[longest_idx], segment_ends[longest_idx]
    longest_duration = segment_lengths[longest_idx]

    print(f"\nLongest contiguous segment: {time_vals[start_idx]:.3f}s to {time_vals[end_idx - 1]:.3f}s "
          f"({longest_duration:.3f} seconds long)")

report_time_gaps(df, time_col='time', gap_threshold=5.0)


Checking for gaps > 5.0 seconds...

Gap size: 2498.999 seconds
Gap size: 744.197 seconds
Gap size: 869.179 seconds
Gap size: 1025.658 seconds
Gap size: 38.464 seconds
Gap size: 55.822 seconds
Gap size: 8.641 seconds
Gap size: 73.150 seconds
Gap size: 279.219 seconds
Gap size: 228.868 seconds
Gap size: 410.528 seconds
Gap size: 145.404 seconds
Gap size: 6.021 seconds
Gap size: 29.699 seconds
Gap size: 36.002 seconds
Gap size: 34.933 seconds
Gap size: 11.518 seconds
Gap size: 6.285 seconds
Gap size: 9.866 seconds
Gap size: 18.137 seconds
Gap size: 5.060 seconds
Gap size: 8.882 seconds
Gap size: 253.200 seconds
Gap size: 10.527 seconds
Gap size: 8.562 seconds
Gap size: 13.703 seconds
Gap size: 15.224 seconds
Gap size: 10.702 seconds
Gap size: 8.085 seconds
Gap size: 5.439 seconds
Gap size: 341.775 seconds
Gap size: 165.650 seconds
Gap size: 57.444 seconds
Gap size: 164.054 seconds
Gap size: 143.785 seconds
Gap size: 258.474 seconds
Gap size: 132.480 seconds
Gap size: 51.779 seconds
Gap s

In [32]:
import plotly.graph_objects as go


# Create a 3D scatter (or line) plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=df['x'],
            y=df['y'],
            z=df['z'],
            mode='lines+markers',  # 'lines', 'markers', or 'lines+markers'
            line=dict(color='royalblue', width=4),
            marker=dict(size=3, color='orange'),
            hovertemplate=
            'Time: %{customdata:.2f}s<extra></extra>',
            customdata=df['time'])
    ]
)

# Add layout details
fig.update_layout(
    title='3D Position Trajectory',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    ),
    height=700
)

fig.show()


Looking at the robot's position over the course of the recovered data, there are large gaps where the robot appears to teleport through space. Some of these are simply missing readings where the robot went AWOL for some time, even up to 2600 seconds, nearly 45 minutes. What is was doing during that time, no one can say. Perhaps it was between runs at that time. 

There are also gaps in space that have only a short gap in time, too short for it to actually travel to that point. This suggesets misordering in the data or else a position encoder error.

In [33]:
import plotly.graph_objects as go

min_points = 83
max_points = 335  # Limit number of points for clarity
# Create a 3D scatter (or line) plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=df['x'][min_points:max_points],
            y=df['y'][min_points:max_points],
            z=df['z'][min_points:max_points],
            mode='lines+markers',  # 'lines', 'markers', or 'lines+markers'
            line=dict(color='royalblue', width=4),
            marker=dict(size=3, color='orange'),
            hovertemplate=
            'Time: %{customdata:.2f}s<extra></extra>',
            customdata=df['time'][min_points:max_points]        )
    ]
)

# Add layout details
fig.update_layout(
    title='3D Position Trajectory',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    ),
    height=700
)

fig.show()


For our purposes we will focus on a short 4-minute interval that contains no major time or space disconinuities.

In [34]:
def detailed_summary(df):
    summary = pd.DataFrame(index=df.columns)

    summary["Data Type"] = df.dtypes
    summary["Unique Values"] = df.nunique()

    numeric_cols = df.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        summary.loc[col, "Mean"] = df[col].mean()
        summary.loc[col, "Median"] = df[col].median()
        summary.loc[col, "Std Dev"] = df[col].std()
        summary.loc[col, "Variance"] = df[col].var()
        summary.loc[col, "Min"] = df[col].min()
        summary.loc[col, "Max"] = df[col].max()
        summary.loc[col, "Skewness"] = df[col].skew()
        summary.loc[col, "Kurtosis"] = df[col].kurt()
        summary.loc[col, "25%"] = df[col].quantile(0.25)
        summary.loc[col, "75%"] = df[col].quantile(0.75)
        summary.loc[col, "IQR"] = summary.loc[col, "75%"] - summary.loc[col, "25%"]

    # Round numeric columns neatly
    summary = summary.round(3)
    return summary

detailed_summary(df)

,Data Type,Unique Values,Mean,Median,Std Dev,Variance,Min,Max,Skewness,Kurtosis,25%,75%,IQR
time,float64,8686,10094.375,10279.160,2782.776,7743840.838,0.000,16533.898,-0.241,-0.849,7957.466,12677.564,4720.098
x,float64,3895,0.013,-0.019,0.179,0.032,-0.333,0.350,0.194,-1.355,-0.142,0.206,0.347
y,float64,1873,0.135,0.186,0.151,0.023,-0.128,0.343,-0.574,-0.981,-0.014,0.264,0.278
z,float64,1322,0.265,0.294,0.059,0.003,0.099,0.319,-1.185,-0.019,0.230,0.308,0.078


In [35]:
def histogram_plots(df_cobots, feature_type_lst=["Current", "Speed", "Temperature"], unit=["A", "m/s", "Degrees C"]):

    colors = px.colors.qualitative.Dark24

    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=[f'{feature} Distribution' for feature in feature_type_lst],
        horizontal_spacing=0.1
    )

    # Loop through each feature type
    for feat_idx, (feature_type, unit_label) in enumerate(zip(feature_type_lst, unit)):
        row = feat_idx + 1  
        
        fig.add_trace(
            go.Histogram(
                x=df_cobots[feature_type],
                marker=dict(color=colors[feat_idx]),
                opacity=0.7,
            ),
            row=row, col=1
        )
        
        fig.update_xaxes(title_text=f"{feature_type} ({unit_label})", row=row, col=1)

    fig.update_yaxes(title_text="Count", row=1, col=1)

    fig.update_layout(
        height=1000,
        barmode='overlay',  
        showlegend=False,

    )
    return fig

feature_type_lst = ["x", "y", "z"]
unit_lst = ["pos", "pos", "pos"]
fig = histogram_plots(df, feature_type_lst=feature_type_lst, unit=unit_lst)
fig.show()

In [36]:
import plotly.graph_objects as go
import numpy as np


 # Calculate correlation
df_corr = df.drop(columns=['time']).corr().round(2)

# Mask upper triangle
mask = np.zeros_like(df_corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

# Apply mask and drop empty rows/cols
df_corr_viz = df_corr.mask(mask).dropna(how='all').dropna(axis='columns', how='all')

# Create text array with blanks instead of nan
text_values = df_corr_viz.values.astype(str)
text_values[text_values == 'nan'] = ''

# Add heatmap to subplot
fig =  go.Heatmap(
        z=df_corr_viz.values,
        x=df_corr_viz.columns,
        y=df_corr_viz.index,
        colorscale='Viridis',
        zmid=0,
        text=text_values,
        texttemplate='%{text}',
        textfont={"size": 8},
    )

fig = go.Figure(data=fig)
fig.update_layout(
    height=400,
    width=425,
    title_text="Correlation Analysis",
    showlegend=False
)
fig.show()

In [37]:
from ikpy.chain import Chain
from ikpy.link import URDFLink, OriginLink
import numpy as np

# Code GPT 5.1
bounds = 9/4
# bounds = 2.0

# --- Define the UR3 robot arm chain ---
UR3_arm = Chain(name="UR3_arm", links=[
    # OriginLink(),  # Base

    # Shoulder Pan
    URDFLink(
        name="shoulder_pan",
        origin_translation=[0, 0, 0.15185],
        origin_orientation=[0, 0, 0],
        rotation=[0, 0, 1],
        # bounds=(-1.65, 1.0)
        # bounds=(-bounds, bounds)
    ),

    # Shoulder Lift
    URDFLink(
        name="shoulder_lift",
        origin_translation=[0, 0.1197, 0],
        origin_orientation=[0, 0, 0],
        rotation=[0, 1, 0],
        # bounds=(-6, -0.0),
        # bounds=(-bounds, bounds)
    ),

    # Elbow
    URDFLink(
        name="elbow",
        origin_translation=[0.24365, 0, 0],
        origin_orientation=[0, 0, 0],
        rotation=[0, 1, 0],
        # bounds=(-2.5, 2.5),
        # bounds=(-bounds, bounds)
    ),

    # Wrist 1
    URDFLink(
        name="wrist_1",
        origin_translation=[0.21325, 0, 0],
        origin_orientation=[0, 0, 0],
        rotation=[0, 1, 0],
        # bounds=(-0.8, 4.0),
        # bounds=(-bounds, bounds)
    ),

    # Wrist 2
    URDFLink(
        name="wrist_2",
        origin_translation=[0, 0.08535, 0],
        origin_orientation=[0, 0, 0],
        rotation=[0, 0, 1],
        # bounds=(-2.5, 3.0),
        # bounds=(-bounds, bounds)
    ),

    # Wrist 3 / End Effector
    URDFLink(
        name="wrist_3",
        origin_translation=[0, 0, 0.0819],
        origin_orientation=[0, 0, 0],
        rotation=[0, 1, 0],
        # bounds=(-1.7, -1.5)
        # bounds=(-bounds, bounds)

    )
])

# --- Define target position (example) ---
target_position = [-0.12, 0.221, 0.166]  # meters (x, y, z)

# Convert to homogeneous transformation matrix (identity rotation)
# target_frame = np.eye(4)
# target_frame[:3, 3] = target_position

# --- Compute inverse kinematics ---
ik_solution = UR3_arm.inverse_kinematics(target_position)

print("Computed joint angles (radians):")
for i, angle in enumerate(ik_solution, 1):  # skip base OriginLink
    print(f"Joint {i}: {angle:.4f} rad")

# --- Verify by computing forward kinematics ---
fk_frame = UR3_arm.forward_kinematics(ik_solution)
print("\nForward Kinematics end-effector position:")
print(fk_frame[:3, 3])


Computed joint angles (radians):
Joint 1: 1.1149 rad
Joint 2: 0.3744 rad
Joint 3: -2.6146 rad
Joint 4: -1.5758 rad
Joint 5: 0.0000 rad
Joint 6: 0.0000 rad

Forward Kinematics end-effector position:
[-0.11999995  0.22099992  0.16600002]


In [38]:
def animation_plot(x, y, z, animation_sequence, robot_chain):
    """Creates a 3D plot of the robot arm given joint positions."""

    # --- Create 3D plot ---
    fig = go.Figure()

    # Robot arm
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='lines+markers',
        line=dict(width=10, color='rgb(59, 130, 246)'),
        marker=dict(size=8, color='rgb(29, 78, 216)'),
        name='Robot Arm',
        showlegend=True
    ))

    # End effector highlight
    fig.add_trace(go.Scatter3d(
        x=[x[-1]], y=[y[-1]], z=[z[-1]],
        mode='markers',
        marker=dict(size=15, color='rgb(255, 255, 255)', 
                    line=dict(width=2, color='rgb(0, 0, 0)')),
        name='End Effector',
        showlegend=True
    ))

    # Base
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode='markers',
        marker=dict(size=10, color='rgb(34, 197, 94)', symbol='diamond'),
        name='Base',
        showlegend=True
    ))


    return fig

In [39]:
import plotly.graph_objects as go
import streamlit as st


angles = ik_solution
frame_matrices = UR3_arm.forward_kinematics([0.0] + angles, full_kinematics=True)
x, y, z = zip(*[m[:3, 3] for m in frame_matrices])

# --- Camera presets ---
camera_presets = {
    "Isometric": dict(x=1.5, y=1.5, z=1.2),
    "Front": dict(x=0.0, y=2.5, z=0.5),
    "Side": dict(x=2.5, y=0.0, z=0.5),
    "Top": dict(x=0.0, y=0.0, z=3.0)
}


# --- Build 3D plot ---
fig = animation_plot(x, y, z, [], UR3_arm)
fig.update_layout(
    autosize=False,
    width=750,             # fixed width
    height=450,
    margin=dict(l=0, r=0, t=0, b=0),
    legend=dict(
            x=0.98, y=0.02, xanchor='left', yanchor='top',
         ),
         scene=dict(
            aspectmode='cube',
            xaxis=dict(title='X', range=[-0.5, 0.5]),
            yaxis=dict(title='Y', range=[-0.5, 0.5]),
            zaxis=dict(title='Z', range=[0, 0.7]),
         ),
         uirevision='static_scene'  # <— prevents re-layout across reruns
      )

fig.show()




In [40]:
# Iterate through df, compute inv kin for each position

previous_solution = None
joint_angles = []
for _, row in df.iterrows():
    target_position= [row.x, row.y, row.z]

    if previous_solution is not None:
        initial_position = previous_solution    
    else:
        initial_position = None

    ik_solution = UR3_arm.inverse_kinematics(
        target_position,
        initial_position=initial_position
    )    

    previous_solution = ik_solution
    joint_angles.append(ik_solution[0:6]) 

In [41]:
# --- Add back to DataFrame ---
joint_cols = [f'q{i}' for i in range(6)]
angles_df = pd.DataFrame(joint_angles, columns=joint_cols)

df = pd.concat([df, angles_df], axis=1)

# df.to_feather('../data/rad/rad.feather')

df

# print(results[0])
# joint_cols = [f'q{i}' for i in range(6)]
# # angles_df = pd.DataFrame(joint_angles, columns=joint_cols)
# angles_df = pd.DataFrame(results, columns=joint_cols)

# df = pd.concat([df, angles_df], axis=1)



,time,x,y,z,q0,q1,q2,q3,q4,q5
0,0.000000,-0.186053,-0.013900,0.25,1.645375,-1.279052,-2.650487e+00,-9.791041e-01,0.0,0.0
1,0.846160,-0.181287,-0.004773,0.25,1.597134,-1.277415,-2.649175e+00,-9.779577e-01,0.0,0.0
2,1.290711,-0.174021,0.002766,0.25,1.554889,-1.277414,-2.649122e+00,-9.779188e-01,0.0,0.0
3,1.717405,-0.165193,0.008004,0.25,1.522375,-1.276783,-2.648628e+00,-9.774757e-01,0.0,0.0
4,2.163429,-0.156365,0.013004,0.25,1.487846,-1.276190,-2.648195e+00,-9.770990e-01,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
11098,16532.287049,-0.154900,0.212000,0.25,470988.874181,-387106.918953,1.458187e+06,-2.357830e+06,0.0,0.0
11099,16532.646601,-0.144900,0.212714,0.25,470988.816562,-387106.946176,1.458186e+06,-2.357830e+06,0.0,0.0
11100,16533.064527,-0.134900,0.213429,0.25,470988.757866,-387106.975416,1.458186e+06,-2.357830e+06,0.0,0.0
11101,16533.480524,-0.124900,0.214619,0.25,470988.697312,-387107.005831,1.458186e+06,-2.357830e+06,0.0,0.0


In [42]:
new_positions = []

for _, row in df.iterrows():
    angles = [row.q0, row.q1, row.q2, row.q3, row.q4, row.q5]
    new_position = UR3_arm.forward_kinematics(angles)[:3, 3]

    new_positions.append(new_position)

new_pos_cols = ['x_hat', 'y_hat', 'z_hat']
new_df = pd.DataFrame(new_positions, columns=new_pos_cols)
df = pd.concat([df, new_df], axis=1)

In [46]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def time_series_plots(df, error=None):
    feature_type_lst = ["x", "y", "z"]
    colors = px.colors.qualitative.Dark24[:3]

    # Create subplots for X, Y, Z
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        subplot_titles=("X Position", "Y Position", "Z Position"),
        vertical_spacing=0.08
    )

    # Loop through each axis
    for idx, feature_type in enumerate(feature_type_lst, start=1):
        color = colors[idx - 1]

        # Plot original position
        if feature_type in df.columns:
            fig.add_trace(
                go.Scatter(
                    x=df['time'],
                    y=df[feature_type],
                    name=f"{feature_type.upper()} (true)",
                    mode='lines',
                    line=dict(color=color, width=2),
                ),
                row=idx, col=1
            )

        # Plot reconstructed / predicted position
        hat_col = f"{feature_type}_hat"
        if hat_col in df.columns:
            fig.add_trace(
                go.Scatter(
                    x=df['time'],
                    y=df[hat_col],
                    name=f"{feature_type.upper()} (pred)",
                    mode='lines',
                    line=dict(color=color, width=2, dash='dash')
                ),
                row=idx, col=1
            )

    # Axis titles
    fig.update_xaxes(title_text="Time (s)", row=3, col=1, rangeslider_visible=True)
    for i, feature_type in enumerate(feature_type_lst, start=1):
        fig.update_yaxes(title_text=f"{feature_type.upper()} (m)", row=i, col=1)

    # Layout
    fig.update_layout(
        height=900,
        title_text="Position Time Series: Actual vs. Forward-Kinematics Reconstruction",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    return fig

# Example usage
fig = time_series_plots(df)
fig.show()


In [50]:
df = pd.read_feather('../data/rad/rad.feather')
df

,time,x,y,z,q0,q1,q2,q3,q4,q5,x_hat,y_hat,z_hat
0,0.000000,-0.186053,-0.013900,0.25,1.645375,-1.279052,-2.650487e+00,-9.791041e-01,0.0,0.0,-0.204480,-0.015277,0.250000
1,0.846160,-0.181287,-0.004773,0.25,1.597134,-1.277415,-2.649175e+00,-9.779577e-01,0.0,0.0,-0.204979,-0.005397,0.250000
2,1.290711,-0.174021,0.002766,0.25,1.554889,-1.277414,-2.649122e+00,-9.779188e-01,0.0,0.0,-0.205024,0.003258,0.250001
3,1.717405,-0.165193,0.008004,0.25,1.522375,-1.276783,-2.648628e+00,-9.774757e-01,0.0,0.0,-0.204810,0.009924,0.250000
4,2.163429,-0.156365,0.013004,0.25,1.487846,-1.276190,-2.648195e+00,-9.770990e-01,0.0,0.0,-0.204345,0.016994,0.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11098,16532.287049,-0.154900,0.212000,0.25,470988.874181,-387106.918953,1.458187e+06,-2.357830e+06,0.0,0.0,-0.154622,0.211748,0.249956
11099,16532.646601,-0.144900,0.212714,0.25,470988.816562,-387106.946176,1.458186e+06,-2.357830e+06,0.0,0.0,-0.144773,0.212576,0.249969
11100,16533.064527,-0.134900,0.213429,0.25,470988.757866,-387106.975416,1.458186e+06,-2.357830e+06,0.0,0.0,-0.134962,0.213495,0.250015
11101,16533.480524,-0.124900,0.214619,0.25,470988.697312,-387107.005831,1.458186e+06,-2.357830e+06,0.0,0.0,-0.124939,0.214662,0.250009
